In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path

if "notebooks" in os.getcwd():
    os.chdir(Path.cwd().parent)

In [ ]:
import logging

logger = logging.getLogger("ts_visualization")
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter("%(levelname)s: %(message)s")
    handler.setFormatter(formatter)
    logger.addHandler(handler)

In [ ]:
import numpy as np

from lets_plot import LetsPlot
from xaitimesynth import (
    TimeSeriesBuilder,
    random_walk,
    gaussian,
    # sine_wave,
    level_change,
    peak,
    plot_class_comparison,
    plot_components,
    shapelet,
    manual,
    plot_component,
    list_components,
    list_signal_components,
    list_feature_components,
)
from xaitimesynth.generators import generate_random_walk
from xaitimesynth.generators import generate_ecg_like


# Enable lets_plot for notebook display
LetsPlot.setup_html()

# Helper functions

# Synth Data

In [ ]:
# see generators/components already defined in package
print(list_components())
print(list_signal_components())
print(list_feature_components())

## Single components

In [ ]:
signal = generate_random_walk(100, np.random.RandomState(42), step_size=0.2)
plot_component(signal)

In [ ]:
plot_component(generate_ecg_like(400, np.random.RandomState(42))).show()
plot_component(component_type="ecg_like", n_timesteps=400, line_color="blue").show()

In [ ]:
# Generate and plot a random walk
plot_component(component_type="random_walk", n_timesteps=100, step_size=0.2).show()

# Generate and plot a seasonal pattern
plot_component(
    component_type="seasonal", n_timesteps=200, period=20, amplitude=1.5
).show()

# Example 1: Using a manual component with custom values

# Create custom values for a zigzag pattern
zigzag = np.concatenate([np.linspace(0, 1, 10), np.linspace(1, 0, 10)])
zigzag = np.tile(zigzag, 5)  # Repeat the pattern 5 times

# Plot using the manual component with custom values
plot_component(
    component_type="manual",
    values=zigzag,
    n_timesteps=100,
    line_color="red",
).show()


# Define a custom generator function for a damped oscillation
def damped_oscillation(length, rng, frequency=0.1, decay=0.05, **kwargs):
    t = np.arange(length)
    return np.exp(-decay * t) * np.sin(2 * np.pi * frequency * t)


# Plot using the manual component with custom generator
plot_component(
    component_type="manual",
    generator=damped_oscillation,
    frequency=0.05,  # Lower frequency
    decay=0.02,  # Slower decay
    n_timesteps=200,
    line_color="blue",
    normalization="minmax",  # Show actual values
).show()

plot_component(
    component_type="manual",
    generator=damped_oscillation,
    frequency=0.05,  # Lower frequency
    decay=0.02,  # Slower decay
    n_timesteps=200,
    line_color="blue",
    normalization="minmax",  # Show actual values
    normalization_kwargs={"feature_range": (-1, 1)},  # Custom normalization range
).show()

## Yaml loading

In [ ]:
import yaml
from xaitimesynth.parser import load_builders_from_config

# Example YAML content as a string
yaml_config_str = """
test_dataset:
  n_timesteps: 100
  n_dimensions: 2
  n_samples: 50 # Added n_samples for builder initialization
  random_state: 42 # Added random_state for reproducibility
  classes:
    - id: 0
      signals:
        - function: random_walk
          params:
            step_size: 0.1
          role: foundation
          dimensions: [0, 1]
        - function: gaussian
          params: {} # sigma defaults to 1.0 if not specified
          role: noise
          # dimensions: null # Defaults to all dimensions if omitted
      features:
        - function: seasonal
          params:
            period: 5
            amplitude: 0.5
          start_pct: 0.2
          end_pct: 0.5
          dimensions: [0]

    - id: 1
      signals:
        - function: random_walk
          params:
            step_size: 0.1
          # dimensions: null # Defaults to all dimensions
      features:
        - function: trend
          params:
            slope: 0.05
          length_pct: 0.3
          random_location: true
          dimensions: [1]
"""

# Load the YAML string into a dictionary
config_dict = yaml.safe_load(yaml_config_str)

# load yaml from filepath
# yaml_file_path = "path/to/your/config.yaml"
# with open(yaml_file_path, "r") as file:
#     config_dict = yaml.safe_load(file)

# --- Create the builder from the dictionary ---
builder_from_config = load_builders_from_config(config_dict)


#### TODO: needs to be adjusted
# # --- Now you can use the builder as usual ---
# dataset = builder_from_config.build()

# # You can also clone it
# train_data = builder_from_config.clone(n_samples=1000, random_state=100).build()
# test_data = builder_from_config.clone(n_samples=500, random_state=200).build()

# print("Builder created successfully from config.")
# print(f"Generated dataset X shape: {dataset['X'].shape}")
# print(f"Train data X shape: {train_data['X'].shape}")
# print(f"Test data X shape: {test_data['X'].shape}")

# # You can still visualize etc.
# # from xaitimesynth import plot_components
# # plot_components(dataset) # Requires lets_plot installed and setup

# for plot in plot_components(train_data):
#     plot.show()

# builder_from_config

In [ ]:
# In a new code cell

import yaml
import os
from pathlib import Path
from xaitimesynth.parser import load_builders_from_config  # Assuming it's in builder.py

# --- Example 1: Loading from a YAML String ---
print("--- Example 1: Loading from YAML String ---")
yaml_config_str = """
test_dataset:
  n_timesteps: 100
  n_dimensions: 2
  n_samples: 50
  random_state: 42
  classes:
    - id: 0
      signals:
        - { function: random_walk, params: { step_size: 0.1 }, role: foundation, dimensions: [0, 1] }
        - { function: gaussian, params: {}, role: noise }
      features:
        - { function: seasonal, params: { period: 5, amplitude: 0.5 }, start_pct: 0.2, end_pct: 0.5, dimensions: [0] }
    - id: 1
      signals:
        - { function: random_walk, params: { step_size: 0.1 } }
      features:
        - { function: trend, params: { slope: 0.05 }, length_pct: 0.3, random_location: true, dimensions: [1] }
"""

# Load builders directly from the string
# Since 'test_dataset' is at the top level, no path_key is needed.
builders_dict_str = load_builders_from_config(config_str=yaml_config_str)

# Access the specific builder by its name
builder_from_str = builders_dict_str["test_dataset"]

# Use the builder
dataset_str = builder_from_str.build()
train_data_str = builder_from_str.clone(n_samples=100, random_state=100).build()
test_data_str = builder_from_str.clone(n_samples=50, random_state=200).build()

print("Builder 'test_dataset' created successfully from YAML string.")
print(f"Generated dataset X shape: {dataset_str['X'].shape}")
print(f"Train data X shape: {train_data_str['X'].shape}")
print(f"Test data X shape: {test_data_str['X'].shape}")
print(f"Available builders loaded: {list(builders_dict_str.keys())}\n")

# Visualize components
plots_str = plot_components(train_data_str)
for p in plots_str:
    p.show()


# --- Example 2: Loading from a YAML File ---
print("\n--- Example 2: Loading from YAML File ---")
# Create a temporary YAML file for demonstration
temp_yaml_content = """
dataset_a:
  n_timesteps: 50
  n_samples: 10
  random_state: 1
  classes:
    - id: 0
      signals: [ { function: constant, params: { value: 0 } } ]

experiments:
  datasets:
    dataset_b:
      n_timesteps: 100
      n_samples: 20
      random_state: 2
      classes:
        - id: 0
          signals: [ { function: random_walk, params: { step_size: 0.1 } } ]
        - id: 1
          signals: [ { function: random_walk, params: { step_size: 0.1 } } ]
          features: [ { function: peak, params: { amplitude: 1.0 }, length_pct: 0.1, random_location: true } ]
    dataset_c:
      n_timesteps: 80
      n_samples: 15
      random_state: 3
      classes:
        - id: 0
          signals: [ { function: seasonal, params: { period: 10 } } ]
"""
temp_yaml_path = Path("temp_config_demo.yaml")
temp_yaml_path.write_text(temp_yaml_content)
print(f"Created temporary config file: {temp_yaml_path.absolute()}")

# Load only 'dataset_a' from the file (top-level)
builder_a_dict = load_builders_from_config(
    config_path=temp_yaml_path, dataset_names=["dataset_a"]
)
print(
    f"Loaded builder 'dataset_a' from file (top-level): {list(builder_a_dict.keys())}"
)
builder_a = builder_a_dict["dataset_a"]
data_a = builder_a.build()
print(f"Dataset A shape: {data_a['X'].shape}")

# Load builders 'dataset_b' and 'dataset_c' from the nested 'experiments/datasets' path
nested_builders = load_builders_from_config(
    config_path=temp_yaml_path, path_key="experiments/datasets"
)
print(
    f"Loaded builders from file (nested path 'experiments/datasets'): {list(nested_builders.keys())}"
)

# Load only 'dataset_c' from the nested path
builder_c_dict = load_builders_from_config(
    config_path=temp_yaml_path,
    path_key="experiments/datasets",
    dataset_names=["dataset_c"],
)
print(
    f"Loaded only builder 'dataset_c' from file (nested path): {list(builder_c_dict.keys())}"
)
builder_c = builder_c_dict["dataset_c"]
data_c = builder_c.build()
print(f"Dataset C shape: {data_c['X'].shape}\n")

# Clean up the temporary file
if temp_yaml_path.exists():
    os.remove(temp_yaml_path)
    print(f"Removed temporary config file: {temp_yaml_path}")


# --- Example 3: Loading from a Python Dictionary ---
print("\n--- Example 3: Loading from Python Dictionary ---")
python_config = {
    "simple_rw": {
        "n_timesteps": 60,
        "n_samples": 5,
        "random_state": 101,
        "classes": [
            {
                "id": 0,
                "signals": [{"function": "random_walk", "params": {"step_size": 0.5}}],
            }
        ],
    },
    "level_shift_example": {
        "n_timesteps": 120,
        "n_samples": 10,
        "random_state": 102,
        "classes": [
            {
                "id": 0,
                "signals": [{"function": "gaussian", "params": {"sigma": 0.1}}],
            },
            {
                "id": 1,
                "signals": [{"function": "gaussian", "params": {"sigma": 0.1}}],
                "features": [
                    {
                        "function": "level_change",
                        "params": {"amplitude": 1.5},
                        "start_pct": 0.6,
                        "end_pct": 0.9,
                    }
                ],
            },
        ],
    },
}

# Load all builders from the dictionary
builders_from_dict = load_builders_from_config(config_dict=python_config)
print(f"Loaded builders from Python dictionary: {list(builders_from_dict.keys())}")

# Access and use one of the builders
ls_builder = builders_from_dict["level_shift_example"]
ls_data = ls_builder.build()
print(f"Level Shift dataset shape: {ls_data['X'].shape}")

# Visualize components
plots_ls = plot_components(ls_data)
plots_ls.show()

In [ ]:
config_dict

## Univariate time series

In [ ]:
from xaitimesynth import seasonal

sine_levelshift_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=200, random_state=42)
    .for_class(0)
    .add_signal(seasonal(period=5), role="foundation")
    .add_feature(seasonal(period=5), start_pct=0.2, end_pct=0.4)
    .add_feature(level_change(amplitude=2.0), start_pct=0.5, end_pct=0.7)
    .for_class(1)
    .add_signal(gaussian(), role="foundation")
    .add_feature(level_change(amplitude=2.0), start_pct=0.5, end_pct=0.7)
    .build()
)
# print(sine_levelshift_dataset.keys())

plot_components(sine_levelshift_dataset).show()

In [ ]:
print(sine_levelshift_dataset.keys())
# print(sine_levelshift_dataset["components"][1].features)

sine_levelshift_dataset["feature_masks"]

In [ ]:
sine_levelshift_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=200, random_state=42)
    .for_class(
        0
    )  # Class 0: Higher frequency sine wave pattern as base signal and level shift
    .add_signal(
        manual(
            generator=lambda n_timesteps, rng: np.sin(
                5 * np.linspace(0, 2 * np.pi, n_timesteps)
            )
        )
    )  # 5 cycles
    .add_feature(level_change(amplitude=2.0), start_pct=0.5, end_pct=0.7)
    .for_class(1)  # Class 1: Constant zero and level shift
    .add_signal(manual(generator=lambda n_timesteps, rng: np.zeros(n_timesteps)))
    .add_feature(level_change(amplitude=2.0), start_pct=0.5, end_pct=0.7)
    .build()
)
print(sine_levelshift_dataset.keys())

plot_components(sine_levelshift_dataset).show()

In [ ]:
sine_levelshift_dataset["X"]

In [ ]:
# Create a dataset with the specified characteristics and higher frequency sine wave
sine_levelshift_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=200, random_state=42)
    .for_class(
        0
    )  # Class 0: Higher frequency sine wave pattern as base signal and level shift
    .add_signal(
        manual(
            generator=lambda n_timesteps, rng: np.sin(
                5 * np.linspace(0, 2 * np.pi, n_timesteps)
            )
        )
    )  # 5 cycles
    .add_feature(level_change(amplitude=2.0), start_pct=0.5, end_pct=0.7)
    .for_class(1)  # Class 1: Constant zero and level shift
    .add_signal(manual(generator=lambda n_timesteps, rng: np.zeros(n_timesteps)))
    .add_feature(level_change(amplitude=2.0), start_pct=0.5, end_pct=0.7)
    .build()
)


custom_order_plot = plot_components(sine_levelshift_dataset)
custom_order_plot.show()

In [ ]:
# Build a dataset
from xaitimesynth import seasonal, trend

dataset_builder = (
    TimeSeriesBuilder(n_timesteps=100, n_dimensions=1)
    # Data for class 0
    .for_class(0)
    .add_signal(random_walk(step_size=0.1), role="foundation")
    .add_signal(gaussian(), role="noise")
    .add_feature(seasonal(period=5, amplitude=0.5), start_pct=0.2, end_pct=0.5)
    # Data for class 1
    .for_class(1)
    .add_signal(random_walk(step_size=0.1))
    .add_feature(trend(slope=0.05), length_pct=0.3, random_location=True)
)

train_data = dataset_builder.clone(n_samples=1000, random_state=100).build()
test_data = dataset_builder.clone(n_samples=500, random_state=200).build()
# # Convert to pandas DataFrame
# builder = TimeSeriesBuilder()
# df = builder.to_df(dataset)

custom_order_plot = plot_components(train_data)
custom_order_plot.show()


# pd.Series(dataset.get("y").flatten()).value_counts(sort=False)

In [ ]:
# Build a dataset
from xaitimesynth import seasonal, trend

dataset_builder = (
    TimeSeriesBuilder(n_timesteps=100, n_dimensions=2)
    # Data for class 0
    .for_class(0)
    .add_signal(random_walk(step_size=0.1), role="foundation", dim=[0, 1])
    .add_signal(gaussian(), role="noise")
    .add_feature(seasonal(period=5, amplitude=0.5), start_pct=0.2, end_pct=0.5, dim=[0])
    # Data for class 1
    .for_class(1)
    .add_signal(random_walk(step_size=0.1))
    .add_feature(trend(slope=0.05), length_pct=0.3, random_location=True, dim=[1])
)

train_data = dataset_builder.clone(n_samples=1000, random_state=100).build()
test_data = dataset_builder.clone(n_samples=500, random_state=200).build()
# # Convert to pandas DataFrame
# builder = TimeSeriesBuilder()
# df = builder.to_df(dataset)

custom_order_plot = plot_components(train_data)

for p in custom_order_plot:
    p.show()


# pd.Series(dataset.get("y").flatten()).value_counts(sort=False)

In [ ]:
"""
Example usage of the XAITimeSynth package.

This example demonstrates how to create a synthetic time series dataset
with two classes, where one has discriminative features and the other doesn't.
"""


# Create a dataset with two classes using the fluent API
dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=200, random_state=42)
    .for_class(0)  # Class 0: No discriminative features
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .for_class(1)  # Class 1: Has discriminative features
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(shapelet(scale=1.0), start_pct=0.4, end_pct=0.6)
    .add_feature(level_change(amplitude=0.5), start_pct=0.7, end_pct=0.9)
    .build()
)

# Print dataset structure
print("Dataset keys:", list(dataset.keys()))
print(f"X shape: {dataset['X'].shape}")
print(f"y shape: {dataset['y'].shape}")
print(f"Classes: {np.unique(dataset['y'])}")
print(f"Class distribution: {np.bincount(dataset['y'])}")
plot_components(dataset).show()

# Example with random feature locations
random_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=200, random_state=42)
    .for_class(0)  # Class 0: No discriminative features
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .for_class(1)  # Class 1: Has discriminative features with random locations
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(shapelet(scale=1.0), random_location=True, length_pct=0.2)
    .add_feature(level_change(amplitude=0.5), random_location=True, length_pct=0.2)
    .build()
)
plot_components(random_dataset).show()

# Example 5: Class comparison alternatives
print("\nClass comparison - default (one panel per class):")
default_comparison = plot_class_comparison(
    random_dataset, panel_width=1000, panel_height=150
)
default_comparison.show()

In [ ]:
# Create a custom component and use it
def my_seasonal_pattern(n_timesteps, rng, frequency=0.1, amplitude=1.0):
    """Generate a custom seasonal pattern."""
    t = np.arange(n_timesteps)
    return amplitude * np.sin(2 * np.pi * frequency * t / n_timesteps)


custom_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=10)
    .for_class(0)
    .add_signal(manual(generator=my_seasonal_pattern, frequency=0.05, amplitude=1.0))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .for_class(1)
    .add_signal(manual(generator=my_seasonal_pattern, frequency=0.05, amplitude=1.0))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(peak(amplitude=2.0, width=5), start_pct=0.4, end_pct=0.6)
    .build()
)

plot_components(custom_dataset).show()

In [ ]:
# Create a dataset with three classes
dataset = (
    TimeSeriesBuilder(
        n_timesteps=100,
        n_samples=30,
        random_state=42,
    )
    # Class 0: Random walk with noise only (baseline)
    .for_class(0)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    # Class 1: Random walk + sine wave feature + level change
    .for_class(1)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(level_change(amplitude=0.7), start_pct=0.7, end_pct=0.9)
    # Class 2: Random walk + peak
    .for_class(2)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(peak(amplitude=2.0, width=5), start_pct=0.4, end_pct=0.6)
    .build()
)


# Example 1: Default visualization with ordered components
# Note: Default order is now "Full Series", "Features", "Base Structure", "Noise"
print("Default visualization with ordered components:")
default_plot = plot_components(dataset)
default_plot.show()


# Example 5: Class comparison alternatives
print("\nClass comparison - default (one panel per class):")
default_comparison = plot_class_comparison(dataset, panel_width=1000, panel_height=150)
default_comparison.show()

## Multivariate time series

In [ ]:
# Create a 3-dimensional time series dataset
multivariate_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=50, n_dimensions=3, random_state=42)
    # Class 0: Random walk in all dimensions but no discriminative features
    .for_class(0)
    .add_signal(random_walk(step_size=0.2), dim=[0, 1, 2])  # Apply to all dimensions
    .add_signal(gaussian(sigma=0.1), role="noise", dim=[0, 1, 2])
    # Class 1: Features in different dimensions
    .for_class(1)
    .add_signal(random_walk(step_size=0.2), dim=[0, 1, 2])
    .add_signal(gaussian(sigma=0.1), role="noise", dim=[0, 1, 2])
    # Add shapelet only to dimensions 0 and 1 with shared location (same position in both)
    .add_feature(
        shapelet(scale=1.2),
        random_location=True,
        length_pct=0.15,
        dim=[0, 1],
        shared_location=True,
    )
    # Add peak only to dimension 2
    .add_feature(
        peak(amplitude=1.0, width=1),
        random_location=True,
        length_pct=0.05,
        dim=[0, 1, 2],
    )
    # Add level change to all dimensions, with different locations for each
    # .add_feature(
    #     level_change(amplitude=0.7),
    #     random_location=True,
    #     length_pct=0.2,
    #     dim=[0, 1, 2],
    #     shared_location=False,
    # )
    .build()
)

print(f"Dataset shape: {multivariate_dataset['X'].shape}")  # Should be (50, 100, 3)
print(f"Number of dimensions: {multivariate_dataset['metadata']['n_dimensions']}")

# Example of accessing a specific dimension
dim0_data = multivariate_dataset["X"][:, :, 0]  # First dimension for all samples
print(f"Dimension 0 shape: {dim0_data.shape}")

# Convert to DataFrame with specific dimensions
df = TimeSeriesBuilder().to_df(multivariate_dataset, dimensions=[0, 1])
print(df.head())

plot_components(multivariate_dataset)

In [ ]:
# Create a 3-dimensional time series dataset
multivariate_dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=50, n_dimensions=3, random_state=42)
    # Class 0: Random walk in all dimensions but no discriminative features
    .for_class(0)
    .add_signal(random_walk(step_size=0.2), dim=[0, 1, 2])  # Apply to all dimensions
    .add_signal(gaussian(sigma=0.1), role="noise", dim=[0, 1, 2])
    # Class 1: Features in different dimensions
    .for_class(1)
    .add_signal(random_walk(step_size=0.2), dim=[0, 1, 2])
    .add_signal(gaussian(sigma=0.1), role="noise", dim=[0, 1, 2])
    # Add shapelet only to dimensions 0 and 1 with shared location
    .add_feature(
        shapelet(scale=1.2),
        random_location=True,
        length_pct=0.15,
        dim=[0, 1],
        shared_location=True,
    )
    # Add peak to all dimensions
    .add_feature(
        peak(amplitude=1.0, width=1),
        random_location=True,
        length_pct=0.05,
        dim=[0, 1, 2],
    )
    .build()
)

# Print dataset information
print(f"Dataset shape: {multivariate_dataset['X'].shape}")  # Should be (50, 100, 3)
print(f"Number of dimensions: {multivariate_dataset['metadata']['n_dimensions']}")

# You can customize the visualization if needed
custom_viz = plot_components(
    multivariate_dataset,
    components=["aggregated", "features"],
    free_y=False,
    panel_width=300,
    panel_height=200,
    dimensions=None,
)
if len(custom_viz) > 1:
    for viz in custom_viz:
        viz.show()
else:
    custom_viz.show()

In [ ]:
# For a single plot or list with one element
viz = plot_components(multivariate_dataset, dimensions=[0])
# viz.show()

# For multiple plots
vizs = plot_components(multivariate_dataset, dimensions=[0, 1])
for i, viz in enumerate(vizs):
    print(f"Dimension {i}:")
    viz.show()

## Creating Train/Test/Validation Splits with clone()

The clone method allows creating multiple datasets from the same builder with different parameters.

In [ ]:
# Create a base builder with class definitions
base_builder = (
    TimeSeriesBuilder(n_timesteps=100, random_state=42)
    .for_class(0)  # Class 0: Just random walk with noise
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    # Class 1: Random walk with noise plus features
    .for_class(1)
    .add_signal(random_walk(step_size=0.2))
    .add_signal(gaussian(sigma=0.1), role="noise")
    .add_feature(shapelet(scale=1.0), start_pct=0.4, end_pct=0.6)
    .add_feature(level_change(amplitude=0.5), start_pct=0.7, end_pct=0.9)
)

train_dataset = base_builder.clone(n_samples=140, random_state=42).build()
test_dataset = base_builder.clone(n_samples=40, random_state=43).build()
val_dataset = base_builder.clone(n_samples=20, random_state=44).build()

print(
    f"Train dataset: X shape={train_dataset['X'].shape}, y shape={train_dataset['y'].shape}"
)
print(
    f"Test dataset: X shape={test_dataset['X'].shape}, y shape={test_dataset['y'].shape}"
)
print(
    f"Validation dataset: X shape={val_dataset['X'].shape}, y shape={val_dataset['y'].shape}"
)

# Compare with explainer indexes
# This is now much simpler since each dataset has its own indexing starting from 0
print("\nSample index 2 in training set is truly sample index 2 (not an offset)")
print("Sample index 2 in test set is truly sample index 2 (not an offset)")

## Metrics

### Correlation Score

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from xaitimesynth.builder import TimeSeriesBuilder
from xaitimesynth.metrics import correlation_score

# Create a simple dataset with seasonal features
dataset = (
    TimeSeriesBuilder(n_timesteps=100, n_samples=2, random_state=42)
    .for_class(0)
    .add_signal(gaussian(mu=0, sigma=0.1), role="foundation")
    .add_feature(seasonal(period=10, amplitude=2.0), start_pct=0.3, end_pct=0.7)
    .build(return_components=True)
)

# Generate different attribution examples
n_timesteps = dataset["metadata"]["n_timesteps"]
n_dimensions = dataset["metadata"]["n_dimensions"]

# Perfect attribution (exactly matches the seasonal feature)
perfect_attr = np.zeros((2, n_timesteps, n_dimensions))
for sample_idx in range(2):
    # Extract the feature component for each sample
    seasonal_feature = None
    for feature_name, feature_values in dataset["components"][
        sample_idx
    ].features.items():
        if "seasonal" in feature_name:
            seasonal_feature = feature_values
            break

    # Copy feature values to attribution (only where features exist - not NaN)
    for t in range(n_timesteps):
        if not np.isnan(seasonal_feature[t]):
            perfect_attr[sample_idx, t, 0] = seasonal_feature[t]

# Inversely correlated attribution
inverse_attr = -1 * perfect_attr

# Random attribution (no correlation with features)
random_attr = np.random.RandomState(42).randn(2, n_timesteps, n_dimensions)

# Calculate correlation scores
absolute_corr_perfect = correlation_score(
    perfect_attr, dataset, feature_source="isolated", absolute=True
)
raw_corr_perfect = correlation_score(
    perfect_attr, dataset, feature_source="isolated", absolute=False
)
absolute_corr_inverse = correlation_score(
    inverse_attr, dataset, feature_source="isolated", absolute=True
)
raw_corr_inverse = correlation_score(
    inverse_attr, dataset, feature_source="isolated", absolute=False
)
random_corr = correlation_score(
    random_attr, dataset, feature_source="isolated", absolute=True
)

# Print the results
print(f"Perfect attribution absolute correlation: {absolute_corr_perfect:.4f}")
print(f"Perfect attribution raw correlation: {raw_corr_perfect:.4f}")
print(f"Inverse attribution absolute correlation: {absolute_corr_inverse:.4f}")
print(f"Inverse attribution raw correlation: {raw_corr_inverse:.4f}")
print(f"Random attribution correlation: {random_corr:.4f}")

# Plot for visual comparison
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
sample_idx = 0

# Extract feature mask
feature_key = next(k for k in dataset["feature_masks"].keys())
mask = dataset["feature_masks"][feature_key][sample_idx]

# Extract original time series
X = dataset["X"][sample_idx, :, 0]

# Extract feature component
seasonal_feature = np.full(n_timesteps, np.nan)
for feature_name, feature_values in dataset["components"][sample_idx].features.items():
    if "seasonal" in feature_name:
        for t in range(n_timesteps):
            if not np.isnan(feature_values[t]):
                seasonal_feature[t] = feature_values[t]

# Plot 1: Original time series with highlighted feature regions
axes[0, 0].plot(X, label="Time Series")
axes[0, 0].fill_between(
    np.arange(n_timesteps),
    np.min(X),
    np.max(X),
    where=mask,
    alpha=0.3,
    color="green",
    label="Feature Region",
)
axes[0, 0].set_title("Time Series with Feature Regions")
axes[0, 0].legend()

# Plot 2: Isolated feature component
axes[0, 1].plot(np.arange(n_timesteps), seasonal_feature, label="Seasonal Feature")
axes[0, 1].set_title("Isolated Feature Component")
axes[0, 1].legend()

# Plot 3: Perfect attribution
axes[1, 0].plot(perfect_attr[sample_idx, :, 0], label="Perfect Attribution")
axes[1, 0].plot(seasonal_feature, "r--", alpha=0.5, label="Feature")
axes[1, 0].set_title(f"Perfect Attribution (corr={raw_corr_perfect:.4f})")
axes[1, 0].legend()

# Plot 4: Inverse attribution
axes[1, 1].plot(inverse_attr[sample_idx, :, 0], label="Inverse Attribution")
axes[1, 1].plot(seasonal_feature, "r--", alpha=0.5, label="Feature")
axes[1, 1].set_title(f"Inverse Attribution (corr={raw_corr_inverse:.4f})")
axes[1, 1].legend()

# Plot 5: Random attribution
axes[2, 0].plot(random_attr[sample_idx, :, 0], label="Random Attribution")
axes[2, 0].plot(seasonal_feature, "r--", alpha=0.5, label="Feature")
axes[2, 0].set_title(f"Random Attribution (corr={random_corr:.4f})")
axes[2, 0].legend()

# Plot 6: Scatter plot of attribution vs feature (in feature regions only)
valid_indices = ~np.isnan(seasonal_feature)
axes[2, 1].scatter(
    seasonal_feature[valid_indices],
    perfect_attr[sample_idx, valid_indices, 0],
    label="Perfect",
    alpha=0.6,
)
axes[2, 1].scatter(
    seasonal_feature[valid_indices],
    inverse_attr[sample_idx, valid_indices, 0],
    label="Inverse",
    alpha=0.6,
)
axes[2, 1].axline([0, 0], [1, 1], linestyle="--", color="gray", label="y=x")
axes[2, 1].axline([0, 0], [1, -1], linestyle="--", color="red", label="y=-x")
axes[2, 1].set_xlabel("Feature Value")
axes[2, 1].set_ylabel("Attribution Value")
axes[2, 1].set_title("Attribution vs. Feature Values")
axes[2, 1].legend()

plt.tight_layout()
plt.show()